<a href="https://colab.research.google.com/github/kruthee05/MACHINELEARNING2/blob/main/candidateElimination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# ---------------------------------------------
# 1. LOAD DATASET FROM CSV FILE
# ---------------------------------------------

csv_path = "/content/consumer_shopping_behavior_survey.csv"

# Read CSV
df = pd.read_csv(csv_path)

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())


# ---------------------------------------------
# 2. SELECT CATEGORICAL FEATURES
# ---------------------------------------------

features = [
    "Gender",
    "Occupation_Status",
    "Shop_Most_Where",
    "Online_Shopping_Frequency",
    "Primary_Device",
    "Follows_Brands_On_Social",
    "Makes_Shopping_List",
    "Preferred_Payment_Method",
    "Uses_BNPL_Installments"
]

target = "Best_Shopping_Mode"


# ---------------------------------------------
# 3. CREATE BINARY TARGET
# Online -> Yes
# Offline / Equal / both -> No
# ---------------------------------------------

data = df[features + [target]].copy()

data["Target"] = data[target].apply(
    lambda x: "Yes" if str(x).strip().lower() == "online" else "No"
)

# Remove original multiclass target
data = data.drop(columns=[target])

# Convert missing values to string
data = data.fillna("Unknown")

print("\nPrepared Dataset:")
print(data.head())


# ---------------------------------------------
# 4. HYPOTHESIS FUNCTIONS
# ---------------------------------------------

def covers(hypothesis, example):
    """
    Check whether a hypothesis covers an example.
    ? means any value.
    """

    for h, x in zip(hypothesis, example):

        if h == "?":
            continue

        if h != x:
            return False

    return True


def more_general_or_equal(h1, h2):
    """
    Returns True if h1 is more general than or equal to h2.
    """

    for a, b in zip(h1, h2):

        if a == "?":
            continue

        if a != b:
            return False

    return True


# ---------------------------------------------
# 5. CANDIDATE ELIMINATION ALGORITHM
# ---------------------------------------------

def candidate_elimination(data, features):

    X = data[features].astype(str).values
    y = data["Target"].values

    n_features = len(features)

    # Most Specific Boundary
    S = None

    # Most General Boundary
    G = [tuple(["?"] * n_features)]

    # Get domain values for every feature
    domains = {}

    for i, feature in enumerate(features):
        domains[i] = list(data[feature].astype(str).unique())

    # -----------------------------------------
    # PROCESS EACH TRAINING EXAMPLE
    # -----------------------------------------

    for example, label in zip(X, y):

        example = list(example)

        # =====================================
        # POSITIVE EXAMPLE
        # =====================================

        if label == "Yes":

            # First positive example
            if S is None:

                S = example.copy()

            else:

                # Remove G hypotheses that
                # do not cover positive example

                G = [
                    g for g in G
                    if covers(g, example)
                ]

                # Generalize S minimally

                for i in range(n_features):

                    if S[i] != example[i]:
                        S[i] = "?"

        # =====================================
        # NEGATIVE EXAMPLE
        # =====================================

        else:

            # Ignore negative examples until
            # we get the first positive example

            if S is None:
                continue

            new_G = []

            for g in G:

                # If g covers negative example,
                # specialize it

                if covers(g, example):

                    for i in range(n_features):

                        # Only specialize ? values
                        if g[i] == "?" and S[i] != "?":

                            # Try values different
                            # from negative example

                            for value in domains[i]:

                                if value != example[i]:

                                    new_hypothesis = list(g)

                                    new_hypothesis[i] = value

                                    new_hypothesis = tuple(
                                        new_hypothesis
                                    )

                                    # Keep only hypotheses
                                    # more general than S

                                    if more_general_or_equal(
                                        new_hypothesis,
                                        S
                                    ):
                                        new_G.append(
                                            new_hypothesis
                                        )

                else:

                    new_G.append(g)

            # Remove duplicates
            G = list(set(new_G))

            # Remove overly specific hypotheses
            final_G = []

            for g in G:

                is_more_specific = False

                for other_g in G:

                    if g != other_g:

                        if (
                            more_general_or_equal(
                                other_g,
                                g
                            )
                        ):
                            is_more_specific = True
                            break

                if not is_more_specific:
                    final_G.append(g)

            G = final_G

    return S, G


# ---------------------------------------------
# 6. RUN ALGORITHM
# ---------------------------------------------

S, G = candidate_elimination(data, features)


# ---------------------------------------------
# 7. DISPLAY RESULTS
# ---------------------------------------------

print("\n" + "=" * 60)
print("SPECIFIC BOUNDARY (S)")
print("=" * 60)

for feature, value in zip(features, S):
    print(f"{feature}: {value}")


print("\n" + "=" * 60)
print("GENERAL BOUNDARY (G)")
print("=" * 60)

for index, hypothesis in enumerate(G, start=1):

    print(f"\nHypothesis {index}:")

    for feature, value in zip(features, hypothesis):
        print(f"{feature}: {value}")

Dataset Shape: (500, 30)

Columns:
['Response_ID', 'Timestamp', 'Age', 'Gender', 'Occupation_Status', 'Country_Region', 'Monthly_Income_Range', 'Shop_Most_Where', 'Online_Shopping_Frequency', 'Preferred_Platform', 'Primary_Device', 'Top_Spending_Category', 'Review_Influence_Score', 'Social_Ads_Influence_Score', 'Price_Comparison_Frequency', 'Discount_Importance_Score', 'Follows_Brands_On_Social', 'Likely_To_Buy_After_Ad_Score', 'Makes_Shopping_List', 'Impulse_Purchase_Frequency', 'Return_Frequency', 'Avg_Monthly_Spend_NonEssentials', 'Preferred_Payment_Method', 'Uses_BNPL_Installments', 'Online_Shopping_Satisfaction_Score', 'Trust_In_Online_Reviews_Score', 'Regret_After_Purchase_Frequency', 'Reason_Prefer_Online', 'Reason_Prefer_InStore', 'Best_Shopping_Mode']

First 5 Rows:
   Response_ID            Timestamp Age  Gender   Occupation_Status  \
0            1  01/08/2026 12:30:25  20    Male                 NaN   
1            2  21/06/2026 13:52:02  24    Male  Employed part-time   
2